<a href="https://colab.research.google.com/github/shivani11-glitch/seizure-detection/blob/main/python_seizure_cnn_extractors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
seizure_cnn_extractors.py
=========================
P3 – 1D CNN Feature Extractors for Multimodal Seizure Detection
----------------------------------------------------------------
Two CNN classes that convert raw physiological signals into feature
maps ready for a Transformer encoder.

Both follow the same contract:
    Input  : (batch_size, in_channels, 1024)   – raw signal window
    Output : (batch_size, 32, 128)             – feature map for Transformer

Architecture strategy
---------------------
We need to shrink the time dimension from 1024 → 128, i.e. an 8× reduction.
We achieve this with THREE stages, each halving the time axis via a
strided convolution (stride=2). Two passes per stage: the first "looks"
at the signal at the current resolution; the second "strides" to
downsample.  Channel depth is progressively widened to 32.
"""

import torch
import torch.nn as nn


# ─────────────────────────────────────────────────────────────────────────────
# Shared helper
# ─────────────────────────────────────────────────────────────────────────────

def _conv_block(in_ch: int, out_ch: int, kernel: int, stride: int) -> nn.Sequential:
    """
    A single convolutional building-block used throughout both CNNs.

    Contains three operations that always travel together:
      Conv1d → BatchNorm1d → ReLU

    Args:
        in_ch  : number of input channels for this block
        out_ch : number of output channels (i.e. how many filters to learn)
        kernel : width of each filter in the time dimension
        stride : step size when sliding the filter; stride=2 halves the length

    The padding formula  padding = kernel // 2  keeps the output length
    the same as the input when stride=1, and halves it cleanly when stride=2
    (given odd-sized kernels like 3 or 5).
    """
    padding = kernel // 2          # "same" padding for odd kernels
    return nn.Sequential(
        nn.Conv1d(
            in_channels  = in_ch,
            out_channels = out_ch,
            kernel_size  = kernel,
            stride       = stride,
            padding      = padding,
            bias         = False,  # bias is redundant when BatchNorm follows
        ),
        nn.BatchNorm1d(out_ch),    # normalises each channel's activations
        nn.ReLU(inplace=True),     # non-linearity; inplace saves a tiny bit of memory
    )


# ─────────────────────────────────────────────────────────────────────────────
# ECG Feature Extractor  (1 channel → 32 channels, 1024 → 128 timesteps)
# ─────────────────────────────────────────────────────────────────────────────

class ECGFeatureExtractor(nn.Module):
    """
    1D CNN for a single-lead ECG signal.

    Expected input  : (batch, 1, 1024)   — one electrode, 1024 samples
    Guaranteed output: (batch, 32, 128)  — 32 feature maps, 128 time steps

    Stage summary
    ─────────────────────────────────────────────────────────────
    Stage   Look conv          Stride conv        Output shape
    ─────────────────────────────────────────────────────────────
    1       (1→8,  k=7, s=1)  (8→8,  k=3, s=2)  (B,  8, 512)
    2       (8→16, k=5, s=1)  (16→16,k=3, s=2)  (B, 16, 256)
    3       (16→32,k=3, s=1)  (32→32,k=3, s=2)  (B, 32, 128)
    ─────────────────────────────────────────────────────────────

    We use a large kernel (k=7) in Stage 1 because ECG has smooth,
    broad waveforms (P wave, QRS complex, T wave); a wide receptive
    field picks them up better from the start.
    """

    def __init__(self):
        super().__init__()

        # ── Stage 1 ──────────────────────────────────────────────────────
        # The signal starts at 1024 samples and has only 1 raw channel.
        # We first 'look' with a wide kernel (k=7) to capture the broad
        # ECG waveform shapes (e.g. the QRS complex spans many samples).
        # Then we stride to compress 1024 → 512.
        self.stage1 = nn.Sequential(
            _conv_block(in_ch=1,  out_ch=8,  kernel=7, stride=1),  # (B, 8,  1024)
            _conv_block(in_ch=8,  out_ch=8,  kernel=3, stride=2),  # (B, 8,  512)
        )

        # ── Stage 2 ──────────────────────────────────────────────────────
        # At 512 samples the signal is already compressed once.
        # We refine with a medium kernel (k=5), then stride again: 512 → 256.
        self.stage2 = nn.Sequential(
            _conv_block(in_ch=8,  out_ch=16, kernel=5, stride=1),  # (B, 16, 512)
            _conv_block(in_ch=16, out_ch=16, kernel=3, stride=2),  # (B, 16, 256)
        )

        # ── Stage 3 ──────────────────────────────────────────────────────
        # Final compression: 256 → 128 and channel expansion to 32.
        # k=3 is the standard kernel at this scale.
        self.stage3 = nn.Sequential(
            _conv_block(in_ch=16, out_ch=32, kernel=3, stride=1),  # (B, 32, 256)
            _conv_block(in_ch=32, out_ch=32, kernel=3, stride=2),  # (B, 32, 128)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : Tensor of shape (batch_size, 1, 1024)
        Returns:
            Tensor of shape (batch_size, 32, 128)
        """
        x = self.stage1(x)  # (B, 1,  1024) → (B, 8,  512)
        x = self.stage2(x)  # (B, 8,  512)  → (B, 16, 256)
        x = self.stage3(x)  # (B, 16, 256)  → (B, 32, 128)
        return x             # ready for Transformer encoder


# ─────────────────────────────────────────────────────────────────────────────
# sEMG Feature Extractor  (4 channels → 32 channels, 1024 → 128 timesteps)
# ─────────────────────────────────────────────────────────────────────────────

class sEMGFeatureExtractor(nn.Module):
    """
    1D CNN for a 4-electrode surface EMG signal.

    Expected input  : (batch, 4, 1024)   — four muscle electrodes, 1024 samples
    Guaranteed output: (batch, 32, 128)  — 32 feature maps, 128 time steps

    Stage summary
    ─────────────────────────────────────────────────────────────
    Stage   Look conv          Stride conv         Output shape
    ─────────────────────────────────────────────────────────────
    1       (4→16, k=5, s=1)  (16→16, k=3, s=2)  (B, 16, 512)
    2       (16→32,k=3, s=1)  (32→32, k=3, s=2)  (B, 32, 256)
    3       (32→32,k=3, s=1)  (32→32, k=3, s=2)  (B, 32, 128)
    ─────────────────────────────────────────────────────────────

    sEMG has 4 channels from the start, so we can immediately widen to
    16 channels in Stage 1 (cheap because in_ch is small). sEMG is also
    higher-frequency than ECG, so we start with k=5 rather than k=7.
    """

    def __init__(self):
        super().__init__()

        # ── Stage 1 ──────────────────────────────────────────────────────
        # Four electrode channels come in simultaneously.
        # The first conv mixes across all 4 channels and learns
        # cross-electrode relationships.  Then stride 1024 → 512.
        self.stage1 = nn.Sequential(
            _conv_block(in_ch=4,  out_ch=16, kernel=5, stride=1),  # (B, 16, 1024)
            _conv_block(in_ch=16, out_ch=16, kernel=3, stride=2),  # (B, 16, 512)
        )

        # ── Stage 2 ──────────────────────────────────────────────────────
        # Expand to 32 channels now — the Transformer's final requirement.
        # Then stride 512 → 256.
        self.stage2 = nn.Sequential(
            _conv_block(in_ch=16, out_ch=32, kernel=3, stride=1),  # (B, 32, 512)
            _conv_block(in_ch=32, out_ch=32, kernel=3, stride=2),  # (B, 32, 256)
        )

        # ── Stage 3 ──────────────────────────────────────────────────────
        # We're already at 32 channels; just compress time: 256 → 128.
        self.stage3 = nn.Sequential(
            _conv_block(in_ch=32, out_ch=32, kernel=3, stride=1),  # (B, 32, 256)
            _conv_block(in_ch=32, out_ch=32, kernel=3, stride=2),  # (B, 32, 128)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : Tensor of shape (batch_size, 4, 1024)
        Returns:
            Tensor of shape (batch_size, 32, 128)
        """
        x = self.stage1(x)  # (B, 4,  1024) → (B, 16, 512)
        x = self.stage2(x)  # (B, 16, 512)  → (B, 32, 256)
        x = self.stage3(x)  # (B, 32, 256)  → (B, 32, 128)
        return x             # ready for Transformer encoder


# ─────────────────────────────────────────────────────────────────────────────
# Testing  —  run this file directly:  python seizure_cnn_extractors.py
# ─────────────────────────────────────────────────────────────────────────────

def test_extractor(model: nn.Module, dummy_input: torch.Tensor,
                   model_name: str, expected_shape: tuple) -> None:
    """
    Runs a single forward pass and checks the output shape.

    Args:
        model         : the CNN instance to test
        dummy_input   : a random tensor simulating a batch of signals
        model_name    : display name for the print output
        expected_shape: the (batch, channels, time) tuple we expect back
    """
    model.eval()                    # disable dropout/batchnorm training behaviour
    with torch.no_grad():           # no gradients needed for a shape check
        output = model(dummy_input) # forward pass through all three stages

    # Build a simple pass/fail message
    shape_ok = output.shape == torch.Size(expected_shape)
    status    = "✅ PASS" if shape_ok else "❌ FAIL"

    print(f"\n{'─'*55}")
    print(f"  Model      : {model_name}")
    print(f"  Input shape: {tuple(dummy_input.shape)}")
    print(f"  Output shape: {tuple(output.shape)}")
    print(f"  Expected   : {expected_shape}")
    print(f"  Result     : {status}")
    print(f"{'─'*55}")

    if not shape_ok:
        # Raise immediately so the mismatch is impossible to miss
        raise AssertionError(
            f"{model_name}: expected {expected_shape}, got {tuple(output.shape)}"
        )


if __name__ == "__main__":

    # ── Configuration ────────────────────────────────────────────────────
    BATCH_SIZE     = 8       # simulate 8 signal windows per forward pass
    SIGNAL_LENGTH  = 1024    # number of time-steps per window
    ECG_CHANNELS   = 1       # single-lead ECG
    SEMG_CHANNELS  = 4       # four surface EMG electrodes

    # Expected output shape (same for both models — Transformer contract)
    EXPECTED_OUT   = (BATCH_SIZE, 32, 128)

    print("\n" + "═"*55)
    print("  Seizure Detection — CNN Feature Extractor Tests")
    print("═"*55)

    # ── ECG test ─────────────────────────────────────────────────────────
    # torch.randn generates random Gaussian noise; shape = (B, 1, 1024)
    # This mimics a batch of raw single-lead ECG windows.
    ecg_dummy  = torch.randn(BATCH_SIZE, ECG_CHANNELS, SIGNAL_LENGTH)
    ecg_model  = ECGFeatureExtractor()
    test_extractor(ecg_model, ecg_dummy, "ECGFeatureExtractor", EXPECTED_OUT)

    # ── sEMG test ────────────────────────────────────────────────────────
    # Shape = (B, 4, 1024): four channels of raw muscle activity.
    semg_dummy = torch.randn(BATCH_SIZE, SEMG_CHANNELS, SIGNAL_LENGTH)
    semg_model = sEMGFeatureExtractor()
    test_extractor(semg_model, semg_dummy, "sEMGFeatureExtractor", EXPECTED_OUT)

    # ── Parameter count ──────────────────────────────────────────────────
    # Good to know before handing off to the Transformer team
    ecg_params  = sum(p.numel() for p in ecg_model.parameters() if p.requires_grad)
    semg_params = sum(p.numel() for p in semg_model.parameters() if p.requires_grad)

    print(f"\n  Trainable parameters")
    print(f"  ECGFeatureExtractor  : {ecg_params:,}")
    print(f"  sEMGFeatureExtractor : {semg_params:,}")
    print("\n  Both extractors output shape (B, 32, 128) — Transformer ready ✅\n")


═══════════════════════════════════════════════════════
  Seizure Detection — CNN Feature Extractor Tests
═══════════════════════════════════════════════════════

───────────────────────────────────────────────────────
  Model      : ECGFeatureExtractor
  Input shape: (8, 1, 1024)
  Output shape: (8, 32, 128)
  Expected   : (8, 32, 128)
  Result     : ✅ PASS
───────────────────────────────────────────────────────

───────────────────────────────────────────────────────
  Model      : sEMGFeatureExtractor
  Input shape: (8, 4, 1024)
  Output shape: (8, 32, 128)
  Expected   : (8, 32, 128)
  Result     : ✅ PASS
───────────────────────────────────────────────────────

  Trainable parameters
  ECGFeatureExtractor  : 6,488
  sEMGFeatureExtractor : 12,160

  Both extractors output shape (B, 32, 128) — Transformer ready ✅



In [ ]:
print(ecg_model)

ECGFeatureExtractor(
  (stage1): Sequential(
    (0): Sequential(
      (0): Conv1d(1, 8, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (1): Sequential(
      (0): Conv1d(8, 8, kernel_size=(3,), stride=(2,), padding=(1,), bias=False)
      (1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
  )
  (stage2): Sequential(
    (0): Sequential(
      (0): Conv1d(8, 16, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (1): Sequential(
      (0): Conv1d(16, 16, kernel_size=(3,), stride=(2,), padding=(1,), bias=False)
      (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
  )
  (sta